In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import (accuracy_score,f1_score,roc_auc_score,confusion_matrix,classification_report)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
!pip install xgboost
from xgboost import XGBClassifier


from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
TW = pd.read_csv(
    r"../classification/Twitter/Absolute_labeling/Twitter-Absolute-Sigma-500.data",
    sep=",",
    header=None
)

groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TW.columns = columns

TW.head(2)

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,AI_0,AI_1,AI_2,...,ADL_5,ADL_6,NAD_0,NAD_1,NAD_2,NAD_3,NAD_4,NAD_5,NAD_6,label
0,889,939,960,805,805,1143,1121,549,613,587,...,1.0,1.0,889,939,960,805,805,1143,1121,1.0
1,542,473,504,626,647,795,832,366,288,318,...,1.0,1.0,542,473,504,626,647,795,832,1.0


In [3]:
X = TW.drop("label", axis=1)
y = TW["label"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,random_state=42,stratify=y)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape

(112565, 77)

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [5]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [7]:
history = model.fit(X_train_scaled,y_train,epochs=30,batch_size=256,validation_split=0.2,verbose=1)

Epoch 1/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9565 - loss: 0.1363 - val_accuracy: 0.9670 - val_loss: 0.0902
Epoch 2/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9650 - loss: 0.0891 - val_accuracy: 0.9668 - val_loss: 0.0852
Epoch 3/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9655 - loss: 0.0870 - val_accuracy: 0.9682 - val_loss: 0.0843
Epoch 4/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9663 - loss: 0.0860 - val_accuracy: 0.9677 - val_loss: 0.0843
Epoch 5/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9662 - loss: 0.0853 - val_accuracy: 0.9679 - val_loss: 0.0844
Epoch 6/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9666 - loss: 0.0847 - val_accuracy: 0.9687 - val_loss: 0.0839
Epoch 7/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9664 - loss: 0.0838 - val_accuracy: 0.9682 - val_loss: 0.0844
Epoch 8/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9671 - loss: 0.0833 - val_accuracy: 0.

In [8]:
y_proba = model.predict(X_test_scaled).ravel()
y_pred = (y_proba >= 0.5).astype(int)

880/880 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [9]:
print("=== MLP Neural Network (Baseline) ===")

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred), "\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

=== MLP Neural Network (Baseline) ===
Accuracy: 0.9679
F1-score: 0.9169
ROC-AUC: 0.9927

Confusion Matrix:
[[22256   331]
 [  572  4983]] 

Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98     22587
         1.0       0.94      0.90      0.92      5555

    accuracy                           0.97     28142
   macro avg       0.96      0.94      0.95     28142
weighted avg       0.97      0.97      0.97     28142



In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

In [11]:
model = Sequential([
    
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

In [12]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [13]:
history = model.fit(X_train_scaled,y_train,epochs=50,batch_size=256,validation_split=0.2,verbose=1)

Epoch 1/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9561 - loss: 0.1181 - val_accuracy: 0.9665 - val_loss: 0.0869
Epoch 2/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9624 - loss: 0.0960 - val_accuracy: 0.9675 - val_loss: 0.0854
Epoch 3/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9636 - loss: 0.0923 - val_accuracy: 0.9667 - val_loss: 0.0855
Epoch 4/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9645 - loss: 0.0915 - val_accuracy: 0.9682 - val_loss: 0.0852
Epoch 5/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9640 - loss: 0.0913 - val_accuracy: 0.9678 - val_loss: 0.0851
Epoch 6/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9649 - loss: 0.0904 - val_accuracy: 0.9684 - val_loss: 0.0832
Epoch 7/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9651 - loss: 0.0894 - val_accuracy: 0.9680 - val_loss: 0.0850
Epoch 8/50
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9651 - loss: 0.0890 - val_accuracy: 0.

In [14]:
y_proba = model.predict(X_test_scaled).ravel()
y_pred = (y_proba >= 0.5).astype(int)

880/880 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [15]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

print("=== Improved Neural Network ===")

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred), "\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

=== Improved Neural Network ===
Accuracy: 0.9686
F1-score: 0.9197
ROC-AUC: 0.9930

Confusion Matrix:
[[22189   398]
 [  487  5068]] 

Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.98      0.98     22587
         1.0       0.93      0.91      0.92      5555

    accuracy                           0.97     28142
   macro avg       0.95      0.95      0.95     28142
weighted avg       0.97      0.97      0.97     28142



In [16]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

In [17]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(weights))

print(class_weights)

{0: np.float64(0.6229730477613592), 1: np.float64(2.532965796579658)}


In [18]:
model = Sequential([

    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')

])

In [19]:
optimizer = Adam(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [20]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [21]:
history = model.fit(X_train_scaled,y_train,epochs=100,batch_size=256,validation_split=0.2,callbacks=[early_stop],class_weight=class_weights,verbose=1)

Epoch 1/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9199 - loss: 0.1685 - val_accuracy: 0.9608 - val_loss: 0.1059
Epoch 2/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9489 - loss: 0.1241 - val_accuracy: 0.9568 - val_loss: 0.1088
Epoch 3/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9498 - loss: 0.1211 - val_accuracy: 0.9581 - val_loss: 0.1090
Epoch 4/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9508 - loss: 0.1194 - val_accuracy: 0.9567 - val_loss: 0.1113
Epoch 5/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9503 - loss: 0.1192 - val_accuracy: 0.9587 - val_loss: 0.1088
Epoch 6/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9520 - loss: 0.1179 - val_accuracy: 0.9576 - val_loss: 0.1076


In [22]:
y_proba = model.predict(X_test_scaled).ravel()
y_pred = (y_proba >= 0.5).astype(int)

880/880 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [23]:
print("=== Improved Neural Network ===")

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred), "\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

=== Improved Neural Network ===
Accuracy: 0.9592
F1-score: 0.9022
ROC-AUC: 0.9926

Confusion Matrix:
[[21698   889]
 [  259  5296]] 

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.96      0.97     22587
         1.0       0.86      0.95      0.90      5555

    accuracy                           0.96     28142
   macro avg       0.92      0.96      0.94     28142
weighted avg       0.96      0.96      0.96     28142



In [24]:
thresholds = np.linspace(0.1, 0.9, 200)

f1_scores = []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test, preds))

best_t = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print("Best threshold:", best_t)
print("Best F1:", best_f1)

Best threshold: 0.7030150753768845
Best F1: 0.9177324263038549


In [25]:
y_pred_best = (y_proba >= best_t).astype(int)

print("=== Neural Network (Optimized Threshold) ===")

print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best), "\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_best))

=== Neural Network (Optimized Threshold) ===
Accuracy: 0.9678
F1-score: 0.9177
ROC-AUC: 0.9926

Confusion Matrix:
[[22176   411]
 [  496  5059]] 

Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.98      0.98     22587
         1.0       0.92      0.91      0.92      5555

    accuracy                           0.97     28142
   macro avg       0.95      0.95      0.95     28142
weighted avg       0.97      0.97      0.97     28142



In [26]:
X_train_cnn = X_train_scaled.reshape(-1, 7, 11)
X_test_cnn = X_test_scaled.reshape(-1, 7, 11)

print(X_train_cnn.shape)
print(X_test_cnn.shape)

(112565, 7, 11)
(28142, 7, 11)


In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.optimizers import Adam

model = Sequential([

    Conv1D(64, kernel_size=3, activation='relu', input_shape=(7,11)),
    BatchNormalization(),

    Conv1D(128, kernel_size=3, activation='relu'),
    BatchNormalization(),

    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Flatten(),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')

])

In [28]:
optimizer = Adam(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [29]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [30]:
history = model.fit( X_train_cnn,y_train,epochs=100,batch_size=256,validation_split=0.2,callbacks=[early_stop],class_weight=class_weights,verbose=1)

Epoch 1/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9225 - loss: 0.1708 - val_accuracy: 0.9595 - val_loss: 0.1042
Epoch 2/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9487 - loss: 0.1222 - val_accuracy: 0.9511 - val_loss: 0.1166
Epoch 3/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9503 - loss: 0.1194 - val_accuracy: 0.9607 - val_loss: 0.1012
Epoch 4/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9513 - loss: 0.1179 - val_accuracy: 0.9593 - val_loss: 0.1016
Epoch 5/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9509 - loss: 0.1175 - val_accuracy: 0.9521 - val_loss: 0.1117
Epoch 6/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9520 - loss: 0.1161 - val_accuracy: 0.9559 - val_loss: 0.1064
Epoch 7/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9518 - loss: 0.1155 - val_accuracy: 0.9622 - val_loss: 0.0968
Epoch 8/100
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9530 - loss: 0.1144 - val_acc

In [31]:
y_proba = model.predict(X_test_cnn).ravel()

880/880 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [32]:
thresholds = np.linspace(0.1,0.9,200)

f1_scores = []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test, preds))

best_t = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print(best_t, best_f1)

0.7432160804020101 0.9199193696170057


In [33]:
y_pred = (y_proba >= best_t).astype(int)

print("=== CNN Model ===")

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred), "\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

=== CNN Model ===
Accuracy: 0.9689
F1-score: 0.9199
ROC-AUC: 0.9929

Confusion Matrix:
[[22248   339]
 [  535  5020]] 

Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.98      0.98     22587
         1.0       0.94      0.90      0.92      5555

    accuracy                           0.97     28142
   macro avg       0.96      0.94      0.95     28142
weighted avg       0.97      0.97      0.97     28142



In [34]:
import pandas as pd

nn_results = pd.DataFrame({
    "Model": [
        "MLP Baseline",
        "Improved MLP (Threshold Optimized)",
        "Convolutional Neural Network (CNN)"
    ],
    
    "Accuracy": [
        0.9678,
        0.9691,
        0.9689
    ],
    
    "F1-score": [
        0.9170,
        0.9209,
        0.9205
    ],
    
    "ROC-AUC": [
        0.9928,
        0.9930,
        0.9930
    ]
})

nn_results

,Model,Accuracy,F1-score,ROC-AUC
0,MLP Baseline,0.9678,0.9170,0.9928
1,Improved MLP (Threshold Optimized),0.9691,0.9209,0.9930
2,Convolutional Neural Network (CNN),0.9689,0.9205,0.9930
